# Assignment 06 · Notebook 04: PyTorch trên KKBox Churn (phân loại)

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Mỗi người dùng là một chuỗi 31 ngày × 8 đặc trưng nghe nhạc; đầu ra là một logit churn. Nhãn lệch (khoảng 9 % churn)
nên dùng `BCEWithLogitsLoss(pos_weight = số âm / số dương)` và chọn ngưỡng trên **validation** theo F1.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import torch
from a06.bench import curves, eval_churn, save_run, sigmoid
from a06.data_churn import load_churn
from a06.export import save_metrics, write_dat
from a06.models_torch import CELLS, CELL_LABELS
from a06.train_torch import run_torch

torch.set_num_threads(16)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

Thiết bị: cuda NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
C = load_churn()
for k in ("train", "val", "test"):
    print(f"{k:5s} {C['X'][k].shape}  tỉ lệ churn {C['y'][k].mean():.4f}")

train (70000, 31, 8)  tỉ lệ churn 0.0896
val   (15000, 31, 8)  tỉ lệ churn 0.0897
test  (15000, 31, 8)  tỉ lệ churn 0.0896


## Huấn luyện trên GPU

Toàn bộ tensor dữ liệu (khoảng 100 MB) nằm sẵn trên VRAM; lô 512, tối đa 20 epoch, dừng sớm sau 4 epoch không cải thiện.

In [3]:
out = run_torch("kkbox", C, device)
meta = save_run("kkbox", "torch", out)

kkbox torch rnn    epoch tốt 15/19 val 1.1389 · 0.59 s/epoch trên cuda


kkbox torch lstm   epoch tốt 17/20 val 1.1257 · 0.60 s/epoch trên cuda


kkbox torch gru    epoch tốt 10/14 val 1.1261 · 0.75 s/epoch trên cuda


kkbox torch bilstm epoch tốt  9/13 val 1.1270 · 2.31 s/epoch trên cuda


## Kết quả trên tập test

In [4]:
res = {c: eval_churn(C, out[c]["pred_val"], out[c]["pred_test"]) for c in CELLS}
pd.DataFrame({CELL_LABELS[c]: {k: v for k, v in res[c].items() if k != "confusion"} for c in CELLS}).T.round(4)

,acc,precision,recall,f1,roc_auc,pr_auc,threshold
Simple RNN,0.8865,0.3483,0.3058,0.3257,0.6703,0.2835,0.6239
LSTM,0.8933,0.3826,0.3103,0.3426,0.6819,0.3102,0.6570
GRU,0.9036,0.4391,0.2738,0.3373,0.6802,0.3004,0.7612
BiLSTM,0.8907,0.3714,0.3170,0.3420,0.6801,0.2975,0.6837


In [5]:
for c in CELLS:
    roc, pr = curves(C["y"]["test"], sigmoid(out[c]["pred_test"]))
    write_dat(f"kkbox_torch_roc_{c}", roc)
    write_dat(f"kkbox_torch_pr_{c}", pr)
    print(CELL_LABELS[c], "ma trận nhầm lẫn [[TN, FP], [FN, TP]] =", res[c]["confusion"])

save_metrics("torch_kkbox", {
    "device": device, "pos_rate_test": float(C["y"]["test"].mean()),
    "models": {c: {**res[c], **{k: meta[f"kkbox_torch_{c}"][k]
                                for k in ("params", "best_epoch", "epochs_run", "epoch_s", "train_s")}}
               for c in CELLS}})
print("đã ghi outputs/metrics/torch_kkbox.json")

Simple RNN ma trận nhầm lẫn [[TN, FP], [FN, TP]] = [[12887, 769], [933, 411]]
LSTM ma trận nhầm lẫn [[TN, FP], [FN, TP]] = [[12983, 673], [927, 417]]
GRU ma trận nhầm lẫn [[TN, FP], [FN, TP]] = [[13186, 470], [976, 368]]
BiLSTM ma trận nhầm lẫn [[TN, FP], [FN, TP]] = [[12935, 721], [918, 426]]
đã ghi outputs/metrics/torch_kkbox.json
